[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/finetuning-avanzado/04-despliegue-modelos-propios.ipynb)

# Despliegue de Modelos Fine-tuneados con vLLM y FastAPI

> Aprende a desplegar modelos LLM propios en producción usando vLLM o TGI,
> construir una API REST con FastAPI y medir el rendimiento con benchmarks.

## Objetivos
- Entender la arquitectura de despliegue de LLMs con vLLM y TGI
- Construir un cliente compatible con la API OpenAI para servidores locales
- Implementar una API wrapper con FastAPI
- Simular benchmarks de rendimiento (tokens/s, latencia)

**Nota:** vLLM y TGI requieren GPU con al menos 16GB VRAM para modelos 7B+.

## 1. Instalación de dependencias

In [ ]:
%pip install fastapi uvicorn requests anthropic pandas matplotlib --quiet

## 2. Arquitectura de despliegue: vLLM y TGI

In [ ]:
import pandas as pd

print("""=== COMANDOS DE ARRANQUE ===""""

# vLLM (recomendado para alta concurrencia)
# pip install vllm
# python -m vllm.entrypoints.openai.api_server \\
#     --model meta-llama/Llama-3.2-3B-Instruct \\
#     --host 0.0.0.0 --port 8080 \\
#     --dtype auto \\
#     --max-model-len 4096

# TGI con Docker (más fácil de desplegar)
# docker run --gpus all -p 8080:80 \\
#     -v $(pwd)/models:/data \\
#     ghcr.io/huggingface/text-generation-inference:latest \\
#     --model-id meta-llama/Llama-3.2-3B-Instruct \\
#     --max-total-tokens 4096
"""
)

# Comparativa vLLM vs TGI
comparativa = pd.DataFrame([
    {"Sistema": "vLLM", "Throughput": "Muy alto", "Latencia": "Baja", "Cuantización": "AWQ, GPTQ, FP8", "Memoria": "PagedAttention", "API": "OpenAI compatible"},
    {"Sistema": "TGI", "Throughput": "Alto", "Latencia": "Baja", "Cuantización": "GPTQ, AWQ, BitsAndBytes", "Memoria": "Continous batching", "API": "Propia + OpenAI"},
    {"Sistema": "Ollama", "Throughput": "Medio", "Latencia": "Media", "Cuantización": "GGUF (llama.cpp)", "Memoria": "Eficiente en CPU", "API": "OpenAI compatible"},
    {"Sistema": "Anthropic API", "Throughput": "Muy alto", "Latencia": "Media", "Cuantización": "N/A", "Memoria": "N/A", "API": "Anthropic"},
])
print("=== COMPARATIVA DE SISTEMAS DE DESPLIEGUE ===")
print(comparativa.to_string(index=False))

## 3. Cliente OpenAI-compatible para vLLM local

In [ ]:
import requests
import time

# vLLM expone una API compatible con OpenAI
# Cuando arranques vLLM, apunta aquí:
VLLM_URL = "http://localhost:8080"
VLLM_MODELO = "meta-llama/Llama-3.2-3B-Instruct"

def llamar_vllm(prompt: str, max_tokens: int = 200) -> dict:
    """Llama a un servidor vLLM con la API compatible con OpenAI."""
    payload = {
        "model": VLLM_MODELO,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.7
    }
    inicio = time.perf_counter()
    try:
        resp = requests.post(f"{VLLM_URL}/v1/chat/completions", json=payload, timeout=30)
        latencia = time.perf_counter() - inicio
        datos = resp.json()
        tokens_salida = datos["usage"]["completion_tokens"]
        return {
            "respuesta": datos["choices"][0]["message"]["content"],
            "tokens_salida": tokens_salida,
            "latencia_s": round(latencia, 3),
            "tokens_por_segundo": round(tokens_salida / latencia, 1)
        }
    except requests.ConnectionError:
        return {"error": "vLLM no disponible. Arranca el servidor primero.",
                "respuesta": "[Servidor no disponible]",
                "tokens_salida": 0, "latencia_s": 0, "tokens_por_segundo": 0}

# Probar la conexión
resultado = llamar_vllm("¿Qué es el machine learning?")
if "error" in resultado:
    print(f"ℹ {resultado['error']}")
    print("Esto es esperado si no tienes vLLM arrancado.")
else:
    print(f"✓ vLLM disponible")
    print(f"Respuesta: {resultado['respuesta'][:200]}")
    print(f"Tokens/segundo: {resultado['tokens_por_segundo']}")

## 4. API wrapper con FastAPI

In [ ]:
# Código de la API FastAPI — guárdalo en api_wrapper.py y ejecútalo con:
# uvicorn api_wrapper:app --host 0.0.0.0 --port 8000

API_WRAPPER_CODE = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import anthropic
import time
from collections import defaultdict

app = FastAPI(title="LLM API Wrapper")
client = anthropic.Anthropic()

# Rate limiting simple en memoria
llamadas_por_usuario = defaultdict(list)
MAX_LLAMADAS_POR_MINUTO = 10

class SolicitudGeneracion(BaseModel):
    prompt: str
    max_tokens: int = 256
    usuario: str = "anonimo"

def verificar_rate_limit(usuario: str) -> bool:
    ahora = time.time()
    # Limpiar llamadas más antiguas de 60 segundos
    llamadas_por_usuario[usuario] = [
        t for t in llamadas_por_usuario[usuario] if ahora - t < 60
    ]
    return len(llamadas_por_usuario[usuario]) < MAX_LLAMADAS_POR_MINUTO

@app.post("/generate")
async def generar(solicitud: SolicitudGeneracion):
    if not verificar_rate_limit(solicitud.usuario):
        raise HTTPException(status_code=429, detail="Rate limit excedido")

    llamadas_por_usuario[solicitud.usuario].append(time.time())

    inicio = time.perf_counter()
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=solicitud.max_tokens,
        messages=[{"role": "user", "content": solicitud.prompt}]
    )
    latencia = time.perf_counter() - inicio

    return {
        "respuesta": resp.content[0].text,
        "tokens_entrada": resp.usage.input_tokens,
        "tokens_salida": resp.usage.output_tokens,
        "latencia_s": round(latencia, 3)
    }

@app.get("/health")
async def health():
    return {"status": "ok"}
'''

with open("api_wrapper.py", "w") as f:
    f.write(API_WRAPPER_CODE)

print("✓ api_wrapper.py generado.")
print("Para arrancar: uvicorn api_wrapper:app --host 0.0.0.0 --port 8000")
print("Para probar: curl -X POST http://localhost:8000/generate -H 'Content-Type: application/json' -d '{\"prompt\": \"Hola\"}'")

## 5. Benchmarks de rendimiento simulados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos de benchmark típicos (basados en mediciones reales publicadas)
sistemas = ["vLLM\nLlama-3.2-3B", "TGI\nMistral-7B", "Ollama\nLlama-3.2-3B", "Anthropic\nHaiku API"]
throughput_tps = [180, 120, 45, 95]      # tokens/segundo
latencia_p50_ms = [350, 480, 1200, 600]  # latencia P50 en ms
latencia_p99_ms = [800, 1100, 3500, 1400] # latencia P99 en ms
coste_por_1m = [0.0, 0.0, 0.0, 1.25]    # USD por 1M tokens salida

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Benchmarks de Despliegue LLM", fontsize=14, fontweight="bold")
colores = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6"]

# Throughput
axes[0].bar(sistemas, throughput_tps, color=colores, edgecolor="black", alpha=0.8)
axes[0].set_title("Throughput (tokens/seg)", fontweight="bold")
axes[0].set_ylabel("Tokens por segundo")
axes[0].tick_params(axis="x", labelsize=8)

# Latencias P50/P99
x = np.arange(len(sistemas))
axes[1].bar(x - 0.2, latencia_p50_ms, 0.4, label="P50", color="#3498db", alpha=0.8)
axes[1].bar(x + 0.2, latencia_p99_ms, 0.4, label="P99", color="#e74c3c", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sistemas, fontsize=8)
axes[1].set_title("Latencia P50 vs P99 (ms)", fontweight="bold")
axes[1].set_ylabel("Milisegundos")
axes[1].legend()

# Coste
axes[2].bar(sistemas, coste_por_1m, color=colores, edgecolor="black", alpha=0.8)
axes[2].set_title("Coste por 1M tokens salida (USD)", fontweight="bold")
axes[2].set_ylabel("USD")
axes[2].tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.savefig("benchmarks_despliegue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica de benchmarks guardada.")